In [1]:
import itertools
from math import exp, factorial

states = []
mdp = {}
mdp1 = {}
mdp2 = {}

actions = {1: "Right", -1: "Left", -4: "Up", 4: "Down"}

goal = tuple([i + 1 for i in range(16)])

theta = 1e-2
gamma = 0.9

In [2]:
def calculate_reward(state, pos, new_pos, row_num):
    temp = state.copy()
    temp[pos] = temp[new_pos]
    temp[new_pos] = 16
    
    target_idx = (row_num + 1) * 4
    if temp[:target_idx] == goal[:target_idx]:
        reward = 75 + (row_num + 1) * 25
    else:
        correct = sum(1 for i in range(target_idx) if temp[i] == goal[i])
        reward = -50 + 2 * correct
    
    return tuple(temp), reward

def generate_transitions(state, empty_pos, row_num):
    transitions = {act: [tuple(state), -1000] for act in actions.values()}
    
    if empty_pos % 4 != 3:
        next_state, rew = calculate_reward(state, empty_pos, empty_pos + 1, row_num)
        transitions[actions[1]] = [next_state, rew]
    
    if empty_pos % 4 != 0:
        next_state, rew = calculate_reward(state, empty_pos, empty_pos - 1, row_num)
        transitions[actions[-1]] = [next_state, rew]
    
    min_row = row_num * 4
    if empty_pos > min_row + 3:
        next_state, rew = calculate_reward(state, empty_pos, empty_pos - 4, row_num)
        transitions[actions[-4]] = [next_state, rew]
    
    if empty_pos < 12:
        next_state, rew = calculate_reward(state, empty_pos, empty_pos + 4, row_num)
        transitions[actions[4]] = [next_state, rew]
    
    return transitions

In [3]:
def generate_configs(start_idx, end_idx, fixed_prefix, values):
    for positions in itertools.combinations(range(start_idx, end_idx), len(values)):
        for perm in itertools.permutations(values):
            config = [0] * 16
            if fixed_prefix:
                config[:len(fixed_prefix)] = fixed_prefix
            for pos, val in zip(positions, perm):
                config[pos] = val
            yield config

for cfg in generate_configs(0, 16, [], [1, 2, 3, 4, 16]):
    states.append(tuple(cfg))
    empty_idx = cfg.index(16)
    mdp[tuple(cfg)] = generate_transitions(cfg, empty_idx, 0)

for cfg in generate_configs(4, 16, list(goal[:4]), [5, 6, 7, 8, 16]):
    states.append(tuple(cfg))
    empty_idx = cfg.index(16)
    mdp1[tuple(cfg)] = generate_transitions(cfg, empty_idx, 1)

for cfg in generate_configs(8, 16, list(goal[:8]), [9, 10, 11, 12, 13, 14, 15, 16]):
    states.append(tuple(cfg))
    empty_idx = cfg.index(16)
    mdp2[tuple(cfg)] = generate_transitions(cfg, empty_idx, 3)

In [4]:
V = {s: 0.0 for s in states}
V[goal] = 1.0
policy = {s: 0 for s in states}

In [5]:
def get_mdp_dict(state):
    zeros = state.count(0)
    if zeros == 0:
        return mdp2
    elif zeros == 7:
        return mdp1
    else:
        return mdp

while True:
    delta = 0
    statevalue = {s: 0.0 for s in states}
    
    for s in states:
        if s == goal:
            statevalue[s] = V[s]
            continue
        
        mdp_dict = get_mdp_dict(s)
        best = float('-inf')
        
        for act in actions.values():
            next_state, reward = mdp_dict[s][act]
            best = max(best, reward + gamma * V[next_state])
        
        delta = max(delta, abs(V[s] - best))
        statevalue[s] = best
    
    V = statevalue
    if delta < theta:
        break

In [6]:
for s in states:
    if s == goal:
        continue
    
    mdp_dict = get_mdp_dict(s)
    best_val = float('-inf')
    best_act = None
    
    for act in actions.values():
        next_state, reward = mdp_dict[s][act]
        val = reward + gamma * V[next_state]
        if val > best_val:
            best_val = val
            best_act = act
    
    policy[s] = best_act